# MNIST Benchmark Analysis

MNIST benchmark analysis on the shared-success subset.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from utils import (
    benchmark_overview,
    build_validity_curve_df,
    load_result,
    method_order,
    method_palette,
    mnist_task_coverage_table,
    mnist_validity_by_target_table,
    plot_heatmap,
    plot_metric_boxplots,
    plot_runtime_bars,
    plot_validity_distance_curves,
    proximity_summary,
    runtime_summary,
    select_methods,
    setup_notebook_style,
    shared_success_retention_summary,
    shared_success_subset,
    style_method_table,
    validity_closeness_score,
    validity_summary,
)

setup_notebook_style()



In [ ]:
RESULT_PATH = Path('../results/benchmark_mnist.parquet')
SELECTED_METHODS = None


In [ ]:
df = load_result(RESULT_PATH)
if SELECTED_METHODS:
    df = select_methods(df, SELECTED_METHODS)

retention = shared_success_retention_summary(df, by=('query_idx',), method_col='method_label')
df = shared_success_subset(df, by=('query_idx',), method_col='method_label')

order = method_order(df)
palette = method_palette(order)

display(retention.style.format({'retention_pct': '{:.1f}%'}))
display(benchmark_overview(df))
print(f'Shared-success rows: {len(df)}')
print(f'Method order: {order}')
df.head(3)


In [ ]:
coverage = mnist_task_coverage_table(df)
display(coverage.style.background_gradient(cmap='Blues', axis=0))
plot_heatmap(coverage, title='Task coverage by source digit and target digit', cmap='Blues', fmt='d', cbar_label='Count')
plt.show()


In [ ]:
validity = validity_summary(df, order=order)
display(style_method_table(validity, fmt={'validity_pct': '{:.1f}%'}, maximize=['validity_pct', 'n_valid']))

validity_by_target = mnist_validity_by_target_table(df).reindex(order)
display(style_method_table(validity_by_target, fmt='{:.1f}%', maximize=list(validity_by_target.columns), gradient_cmap='YlGn'))
plot_heatmap(validity_by_target, title='Validity by target digit (%)', cmap='YlGn', fmt='.1f', cbar_label='Validity (%)', vmin=0, vmax=100)
plt.show()


In [ ]:
curve_l2 = build_validity_curve_df(df, metric='l2_distance', group_cols=('method_label',))
curve_l1 = build_validity_curve_df(df, metric='l1_distance', group_cols=('method_label',))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_validity_distance_curves(curve_l2, palette=palette, title='MNIST validity-distance curve (L2)', ax=axes[0])
plot_validity_distance_curves(curve_l1, palette=palette, title='MNIST validity-distance curve (L1)', ax=axes[1])
fig.tight_layout()
plt.show()


In [ ]:
proximity = proximity_summary(df, order=order, success_only_rows=False)
display(style_method_table(proximity, fmt='{:.3f}', minimize=['mean_l2', 'median_l2', 'mean_l1', 'median_l1', 'mean_mad_l1', 'mean_sparsity', 'mean_redundancy']))
plot_metric_boxplots(
    df,
    metrics=[
        ('l2_distance', 'L2 distance'),
        ('l1_distance', 'L1 distance'),
        ('l0_sparsity', 'Fraction of changed pixels'),
    ],
    method_order=order,
    palette=palette,
    title='MNIST proximity and sparsity',
)
plt.show()


In [ ]:
rt_summary = runtime_summary(df, order=order)
display(style_method_table(rt_summary, fmt='{:.3f}', minimize=['build_time_s', 'mean_query_time_s', 'total_query_time_s']))
plot_runtime_bars(rt_summary, palette=palette)
plt.show()


In [ ]:
finite_l2 = df.loc[df['success'], 'l2_distance'].to_numpy(dtype=float)
budget_l2 = float(np.quantile(finite_l2, 0.95)) if finite_l2.size else 1.0
vcs_rows = []
for method in order:
    sub = df[df['method_label'] == method]
    distances = sub['l2_distance'].to_numpy(dtype=float)
    distances[~sub['success'].to_numpy(dtype=bool)] = np.inf
    vcs_rows.append({'method': method, 'vcs_l2': validity_closeness_score(distances, budget=budget_l2)})

vcs_summary = pd.DataFrame(vcs_rows).set_index('method')
display(style_method_table(vcs_summary, fmt='{:.3f}', maximize=['vcs_l2'], gradient_cmap='Purples'))
